# 23 · Agents, tool calling, and bounded execution

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Validate tool arguments; implement a read-only tool registry; optionally run a real tool-calling loop.


## Step 1 — Define a narrow tool

Tools should expose small structured operations. Never execute model-generated Python, shell commands or SQL directly.


In [ ]:
import json
orders={'A100':{'status':'shipped'},'A200':{'status':'processing'}}
def lookup_order(order_id):
    if not isinstance(order_id,str) or order_id not in orders:
        raise ValueError('Unknown course order id')
    return dict(order_id=order_id,**orders[order_id])
def dispatch(name,arguments):
    if name!='lookup_order': raise ValueError('Tool not allowed')
    data=json.loads(arguments)
    if not isinstance(data,dict) or set(data)!={'order_id'}: raise ValueError('Invalid argument keys')
    return lookup_order(data['order_id'])
print(dispatch('lookup_order','{"order_id":"A100"}'))


**Check your result:** Only allowlisted reads over course sample data are possible.


## Step 2 — Describe the schema

The model proposes a call; your application validates and executes it. This schema alone does not provide authorization.


In [ ]:
tools=[{'type':'function','function':{'name':'lookup_order','description':'Read the status of a course demo order.',
 'parameters':{'type':'object','properties':{'order_id':{'type':'string','enum':['A100','A200']}},'required':['order_id'],'additionalProperties':False}}}]
for name,args in [('shell','{}'),('lookup_order','{"order_id":"A999"}')]:
    try: dispatch(name,args)
    except ValueError: pass
    else: raise AssertionError('Unsafe request accepted')
print('Invalid calls rejected')


**Check your result:** Unknown tools and invalid order IDs are rejected.


## Step 3 — Initialize a real model client optionally

Use a chat endpoint that supports function/tool calling. Default execution tests only the tool dispatcher; it is not presented as model reasoning.


In [ ]:
# OFF by default. Select an existing chat endpoint available in your workspace.
RUN_LIVE = False
CHAT_ENDPOINT = ''
if RUN_LIVE:
    assert CHAT_ENDPOINT, 'Set CHAT_ENDPOINT from the Serving UI.'
    from databricks.sdk import WorkspaceClient
    client = WorkspaceClient().serving_endpoints.get_open_ai_client()

messages=[{'role':'system','content':'Answer order-status questions using lookup_order. Never invent a status.'},
          {'role':'user','content':'What is the status of order A100?'}]


**Check your result:** The live client is initialized only when enabled.


## Step 4 — Run a bounded agent loop

Limit rounds and calls. Tool errors become visible tool results so the model can ask for clarification; no automatic writes are exposed.


In [ ]:
if RUN_LIVE:
    total_calls=0
    for turn in range(3):
        response=client.chat.completions.create(model=CHAT_ENDPOINT,messages=messages,tools=tools,max_tokens=250,temperature=0)
        message=response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))
        if not message.tool_calls:
            print(message.content)
            break
        for call in message.tool_calls:
            total_calls+=1
            if total_calls>4: raise RuntimeError('Tool budget exceeded')
            try: result=dispatch(call.function.name,call.function.arguments)
            except (ValueError,TypeError) as exc: result={'error':str(exc)}
            messages.append({'role':'tool','tool_call_id':call.id,'content':json.dumps(result)})
    else:
        raise RuntimeError('Agent stopped at round limit without a final answer')
else:
    print('Dispatcher tests complete; live model agent skipped.')


**Check your result:** Enabled mode either returns a final answer or stops with a bounded, visible error.


## Practice

Add a read-only product-price lookup with its own JSON schema. Test unknown tools, malformed arguments and repeated calls.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Validate each tool independently. Add per-user record authorization before exposing real orders; a valid order ID does not prove the caller can access it.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
